### Import Libraries

In [ ]:
import unicodedata

import pandas as pd
import numpy as np

import joblib

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
model = joblib.load("../models/f1_xgboost_model.pkl")
FEATURES = joblib.load("../models/model_features.pkl")

### Load Historical Feature Dataset

We use this to:

- calculate rolling features
- compute momentum
- update standings

In [ ]:
df = pd.read_csv("../data/processed/f1_eda_ready.csv")
df.head()

### Load Driver/Constructor Encodings

We must use SAME encodings as training.

In [ ]:
# Load encoders
le_driver = joblib.load("../models/le_driver.pkl")
le_constructor = joblib.load("../models/le_constructor.pkl")
le_circuit = joblib.load("../models/le_circuit.pkl")
print("Encoders loaded.")

### Name Matching Helpers

Driver and team names are written differently in different sources
(`Hulkenberg` vs `Hülkenberg`, `Red Bull Racing` vs `Red Bull`, `Racing Bulls` vs `RB F1 Team`).
If a name does not match, the model gets NO history for that driver, so we match names automatically.

In [ ]:
def _norm(text):
    """lower-case, no accents, no extra spaces:  'Hülkenberg ' -> 'hulkenberg'"""
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()
    return text.strip().lower()


# Team names change between sources.
# Left = the name you type / FastF1 uses.  Right = names to look for in our data (first one found is used).
RB_TEAM = ["RB F1 Team", "RB", "AlphaTauri", "Toro Rosso"]
TEAM_ALIASES = {
    "red bull racing": ["Red Bull"],
    "racing bulls": RB_TEAM,
    "rb": RB_TEAM,
    "kick sauber": ["Sauber", "Alfa Romeo"],
}


def resolve_name(name, known):
    """Return the spelling used in `known` that matches `name`. If nothing matches, return the cleaned name."""
    clean = str(name).strip()
    lookup = {_norm(k): k for k in known}
    key = _norm(clean)

    if key in lookup:                                  # same name (ignoring accents / spaces / case)
        return lookup[key]
    for alias in TEAM_ALIASES.get(key, []):            # known team renames
        if _norm(alias) in lookup:
            return lookup[_norm(alias)]
    if len(key) >= 4:                                  # partial match: 'Haas' -> 'Haas F1 Team'
        for k_norm, k in lookup.items():
            if key in k_norm or k_norm in key:
                return k
    return clean


def to_surname(full_name, known):
    """'Max Verstappen' -> 'Verstappen' (the format used in our training data)."""
    parts = str(full_name).strip().split()
    lookup = {_norm(k): k for k in known}
    for n in (2, 1):                                   # try 'de Vries' first, then 'Vries'
        candidate = _norm(" ".join(parts[-n:]))
        if candidate in lookup:
            return lookup[candidate]
    return parts[-1] if parts else str(full_name)

In [ ]:
# Optional: see the exact names the model knows
print(sorted(df["driver_name"].unique()))
print(sorted(df["constructor_name"].unique()))

### Add 2026 Results (recent form)

The training data stops at 2024. This adds the 2026 races you downloaded earlier so that
"recent form" and "points this season" are real 2026 numbers.
If the file is missing, the notebook still works using 2018-2024 only.

In [ ]:
SEASON = 2026
KEEP = ["year", "round", "driver_name", "constructor_name", "finish_position", "points", "circuit_id"]

try:
    f1_2026 = pd.read_csv("../data/processed/f1_2026_results.csv")
except (FileNotFoundError, pd.errors.EmptyDataError):
    f1_2026 = pd.DataFrame()

ROUND_OF = {}

if len(f1_2026) > 0:
    hist_drivers = set(df["driver_name"].unique())
    hist_teams = set(df["constructor_name"].unique())

    # make 2026 names look like the training data
    f1_2026["driver_name"] = f1_2026["driver_name"].apply(lambda n: to_surname(n, hist_drivers))
    f1_2026["constructor_name"] = f1_2026["constructor_name"].apply(lambda n: resolve_name(n, hist_teams))

    # round number = order the races appear in the file
    race_order = list(dict.fromkeys(f1_2026["race_name"]))
    ROUND_OF = {name: i + 1 for i, name in enumerate(race_order)}
    f1_2026["round"] = f1_2026["race_name"].map(ROUND_OF)
    f1_2026["year"] = SEASON
    f1_2026["finish_position"] = pd.to_numeric(f1_2026["finish_position"], errors="coerce").fillna(20)
    f1_2026["points"] = pd.to_numeric(f1_2026["points"], errors="coerce").fillna(0)
    f1_2026["circuit_id"] = np.nan

    all_df = pd.concat([df[KEEP], f1_2026[KEEP]], ignore_index=True)
    print("2026 races loaded:", race_order)
else:
    all_df = df[KEEP].copy()
    print("No 2026 results found - using 2018-2024 only.")

all_df = all_df.sort_values(["year", "round"]).reset_index(drop=True)


def history_before(year, race_name=None):
    """All races that happened BEFORE the race we want to predict (so the answer is never in the history)."""
    rnd = ROUND_OF.get(race_name, 999)
    return all_df[(all_df["year"] < year) | ((all_df["year"] == year) & (all_df["round"] < rnd))]

### Prediction Function

In [ ]:
def build_race_features(quali_df, history_df, circuit_id, season=SEASON):
    """
    Build the same features the model was trained on.
      quali_df   : one row per driver -> driver_name, constructor_name, grid_position, quali_position
      history_df : ONLY races before the race being predicted (use history_before)
      circuit_id : e.g. "monaco"
      season     : year of the race (used for "points so far THIS season")
    """
    history_df = history_df.sort_values(["year", "round"])
    known_drivers = set(history_df["driver_name"].dropna().unique())
    known_teams = set(history_df["constructor_name"].dropna().unique())

    enc_drivers = set(le_driver.classes_)
    enc_teams = set(le_constructor.classes_)
    enc_circuits = set(le_circuit.classes_)

    rows, no_history, unknown_drivers, unknown_teams = [], [], [], []

    for _, row in quali_df.iterrows():
        driver = resolve_name(row["driver_name"], known_drivers)
        team = resolve_name(row["constructor_name"], known_teams)

        d = history_df[history_df["driver_name"] == driver]
        c = history_df[history_df["constructor_name"] == team]
        d_season = d[d["year"] == season]
        c_season = c[c["year"] == season]

        # Recent form: last 3 races
        avg_finish_last3 = d["finish_position"].tail(3).mean()
        avg_points_last3 = d["points"].tail(3).mean()
        constructor_avg_points_last3 = c.groupby(["year", "round"])["points"].sum().tail(3).mean()

        # Reliability: last 5 races (same as training)
        dnf_rate = (d["finish_position"].tail(5) > 15).mean() if len(d) > 0 else 0

        # Driver record at this circuit
        circuit_hist = d[d["circuit_id"] == circuit_id]
        win_rate = (circuit_hist["finish_position"] == 1).mean() if len(circuit_hist) > 0 else 0

        # Championship points THIS season only (same as training)
        driver_points_so_far = d_season["points"].sum()
        constructor_points_so_far = c_season["points"].sum()

        # Encodings (-1 = the saved encoder has never seen this name)
        driver_encoded = le_driver.transform([driver])[0] if driver in enc_drivers else -1
        constructor_encoded = le_constructor.transform([team])[0] if team in enc_teams else -1
        circuit_encoded = le_circuit.transform([circuit_id])[0] if circuit_id in enc_circuits else -1

        if len(d) == 0:
            no_history.append(row["driver_name"])
        if driver not in enc_drivers:
            unknown_drivers.append(row["driver_name"])
        if team not in enc_teams and row["constructor_name"] not in unknown_teams:
            unknown_teams.append(row["constructor_name"])

        rows.append({
            "driver_name": row["driver_name"],
            "constructor_name": row["constructor_name"],
            "grid_position": row["grid_position"],
            "quali_position": row["quali_position"],
            "driver_avg_finish_last3": avg_finish_last3,
            "driver_avg_points_last3": avg_points_last3,
            "constructor_avg_points_last3": constructor_avg_points_last3,
            "driver_win_rate_at_circuit": win_rate,
            "driver_dnf_rate": dnf_rate,
            "driver_points_so_far": driver_points_so_far,
            "constructor_points_so_far": constructor_points_so_far,
            "driver_encoded": driver_encoded,
            "constructor_encoded": constructor_encoded,
            "circuit_encoded": circuit_encoded,
        })

    prediction_df = pd.DataFrame(rows)

    # Fill missing values the same way as training
    prediction_df["driver_avg_finish_last3"] = prediction_df["driver_avg_finish_last3"].fillna(20)
    remaining = ["driver_avg_points_last3", "constructor_avg_points_last3", "driver_win_rate_at_circuit",
                 "driver_dnf_rate", "driver_points_so_far", "constructor_points_so_far"]
    prediction_df[remaining] = prediction_df[remaining].fillna(0)

    # Tell the user about names that did not match
    if no_history:
        print("No race history found for:", no_history)
    if unknown_drivers:
        print("Drivers the saved model has never seen (encoded as -1):", unknown_drivers)
    if unknown_teams:
        print("Teams the saved model has never seen (encoded as -1):", unknown_teams)

    return prediction_df


def predict_race(features):
    """Win probability for each driver, scaled so the whole race adds up to 100%."""
    raw = model.predict_proba(features[FEATURES])[:, 1]
    result = features[["driver_name", "constructor_name", "grid_position"]].copy()
    result["win_probability"] = raw / raw.sum()
    return result.sort_values("win_probability", ascending=False).reset_index(drop=True)

### Example

In [ ]:
spain_quali = pd.DataFrame({
    "driver_name": [
        "Russell", "Hamilton", "Antonelli", "Norris", "Verstappen", "Hadjar",
        "Piastri", "Lawson", "Hulkenberg", "Leclerc", "Lindblad", "Bortoleto",
        "Colapinto", "Gasly", "Bearman", "Sainz", "Ocon", "Albon",
        "Perez", "Bottas", "Stroll", "Alonso"
    ],
    "constructor_name": [
        "Mercedes", "Ferrari", "Mercedes", "McLaren", "Red Bull", "Red Bull",
        "McLaren", "Racing Bulls", "Audi", "Ferrari", "Racing Bulls", "Audi",
        "Alpine", "Alpine", "Haas", "Williams", "Haas", "Williams",
        "Cadillac", "Cadillac", "Aston Martin", "Aston Martin"
    ],
    "grid_position":  list(range(1, 23)),
    "quali_position": list(range(1, 23))
})

In [ ]:
spain_features = build_race_features(
    quali_df=spain_quali,
    history_df=history_before(2026, "Spain"),
    circuit_id="catalunya"
)
spain_features.head()

### Predict Win Probabilities

In [ ]:
FEATURES

In [ ]:
predictions = predict_race(spain_features)
predictions

In [ ]:
winner = predictions.iloc[0]
print("🏆 Predicted Winner")
print(f"{winner['driver_name']} ({winner['constructor_name']})")
print(f"Win Probability: {winner['win_probability']:.2%}")

### Testing on completed race

#### Step 1 — Create Monaco Qualifying Data

In [ ]:
monaco_quali = pd.DataFrame({
    "driver_name": [
        "Antonelli", "Verstappen", "Hamilton", "Leclerc", "Hadjar",
        "Russell", "Piastri", "Norris", "Gasly", "Lawson"
    ],
    "constructor_name": [
        "Mercedes", "Red Bull", "Ferrari", "Ferrari", "Red Bull",
        "Mercedes", "McLaren", "McLaren", "Alpine", "Racing Bulls"
    ],
    "grid_position":  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "quali_position": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
})
monaco_quali

#### Step 2 — Build Features

In [ ]:
# history_before(2026, "Monaco") only uses races BEFORE Monaco, so the result is not leaked
monaco_features = build_race_features(
    quali_df=monaco_quali,
    history_df=history_before(2026, "Monaco"),
    circuit_id="monaco"
)
monaco_features

#### Step 3 — Predict Monaco

In [ ]:
monaco_predictions = predict_race(monaco_features)
monaco_predictions

#### Step 4 — Compare with the real result

In [ ]:
actual_results = pd.DataFrame({
    "driver_name": ["Antonelli", "Hamilton", "Hadjar"],
    "actual_finish": [1, 2, 3]
})

comparison = monaco_predictions.merge(actual_results, on="driver_name", how="left")
display(comparison)

predicted_top3 = monaco_predictions.head(3)["driver_name"].tolist()
actual_winner = actual_results.loc[actual_results["actual_finish"] == 1, "driver_name"].iloc[0]
print("Predicted top 3:", predicted_top3)
print("Actual winner  :", actual_winner)
print("Winner inside predicted top 3:", actual_winner in predicted_top3)